In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.tensorboard import SummaryWriter
from collections import deque
from torch.distributions import Categorical
import random
import time
import sys

sys.path.append('/kaggle/input/datasets/rks2047/obelix-2')
from obelix import OBELIX
from collections import deque, defaultdict
from torch.distributions import Categorical
import math


In [ ]:
class RecurrentPPO(nn.Module):
    def __init__(self, action_dim, hidden_size=64):
        super(RecurrentPPO, self).__init__()
        self.hidden_size = hidden_size
        self.action_dim = action_dim

        self.input_layer = nn.Linear(action_dim, hidden_size)

        self.gru = nn.GRU(hidden_size, hidden_size, batch_first=True)

        self.actor = nn.Linear(hidden_size, action_dim)
        self.critic = nn.Linear(hidden_size, 1)

    def forward(self, prev_action, hidden):
        prev_action_onehot = F.one_hot(prev_action, num_classes=self.action_dim).float()
        x = F.relu(self.input_layer(prev_action_onehot))
        x = x.unsqueeze(1)
        x, next_hidden = self.gru(x, hidden)
        x = x.squeeze(1)

        action_logits = self.actor(x)
        state_value = self.critic(x)

        return action_logits, state_value, next_hidden

In [ ]:
def train_unwedge_agent(env, model, epochs=1000, gamma=0.99, eps_clip=0.2, update_every=5):
    ACTIONS = ("L45", "L22", "FW", "R22", "R45")
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device)
    optimizer = optim.Adam(model.parameters(), lr=5e-5)
    recent_averages = deque(maxlen=10)
    best_moving_avg = 100

    optimizer.zero_grad()
    stuck_events_since_update = 0

    for epoch in range(epochs):
        state = env.reset()
        done = False

        ep_stuck_attempts = 0
        ep_stuck_steps = 0

        while not done:
            if state[17] == 0:
                next_state, _, done = env.step(ACTIONS[2], render=False)
                state = next_state

            else:
                ep_stuck_attempts += 1

                hidden = torch.zeros(1, 1, model.hidden_size).to(device)
                prev_action = torch.tensor([0]).to(device)

                log_probs, values, rewards = [], [], []
                unwedged = False

                while not done and not unwedged:
                    logits, value, hidden = model(prev_action, hidden)

                    probs = F.softmax(logits, dim=-1)
                    dist = torch.distributions.Categorical(probs)
                    action = dist.sample()

                    next_state, _, done = env.step(ACTIONS[action.item()], render=False)

                    if next_state[17] == 1:
                        reward = -1.0
                    else:
                        reward = 10.0
                        unwedged = True

                    log_probs.append(dist.log_prob(action))
                    values.append(value)
                    rewards.append(reward)

                    state = next_state
                    prev_action = action
                    ep_stuck_steps += 1

                if len(rewards) > 0:
                    returns = []
                    R = 0
                    for r in reversed(rewards):
                        R = r + gamma * R
                        returns.insert(0, R)

                    returns = torch.tensor(returns).to(device)

                    log_probs_stack = torch.stack(log_probs)
                    values_stack = torch.stack(values).view(-1)
                    returns = returns.view(-1)

                    advantages = returns - values_stack.detach()
                    actor_loss = -(log_probs_stack * advantages).mean()
                    critic_loss = F.mse_loss(values_stack, returns)

                    loss = (actor_loss + 0.5 * critic_loss) / update_every
                    loss.backward()

                    stuck_events_since_update += 1

                    if stuck_events_since_update >= update_every:
                        torch.nn.utils.clip_grad_norm_(model.parameters(), 0.5)
                        optimizer.step()
                        optimizer.zero_grad()
                        stuck_events_since_update = 0

        if stuck_events_since_update > 0:
            torch.nn.utils.clip_grad_norm_(model.parameters(), 0.5)
            optimizer.step()
            optimizer.zero_grad()
            stuck_events_since_update = 0

        if ep_stuck_attempts > 0:
            epoch_avg = ep_stuck_steps / ep_stuck_attempts
            recent_averages.append(epoch_avg)

        if len(recent_averages) == 10:
            moving_avg = sum(recent_averages) / 10.0

            print(f"Epoch {epoch} | ep_stuck_attempts {ep_stuck_attempts} | 10-Ep Moving Avg: {moving_avg:.1f} steps")

            if moving_avg < best_moving_avg:
                best_moving_avg = moving_avg
                torch.save(model.state_dict(), "best_unwedge_gru_latest.pth")
                print(f"-> New best stable model saved! ({best_moving_avg:.1f} steps)")

In [ ]:
from obelix import OBELIX
env = OBELIX(scaling_factor = 5,
             arena_size = 500,
             max_steps = 2000,
             wall_obstacles = True,
             seed = 0,
             difficulty = 0)

model = RecurrentPPO(5)

train_unwedge_agent(env, model, epochs=1000)